In [ ]:
# ONE-CELL — Wound AI: CNN (no augmentation) + Leak-safe SBERT Fusion
# Steps:
# 1) Run this cell.
# 2) When prompted, upload your wound dataset zip (folders = wound classes).
# 3) Everything else is automatic

!pip install -q sentence-transformers matplotlib seaborn joblib scikit-learn

import os, shutil, zipfile, re, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Input, Dense, Conv2D, MaxPooling2D, BatchNormalization,
    Flatten, Dropout, Concatenate
)
from tensorflow.keras.utils import to_categorical
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import joblib

from google.colab import files

SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

# =========================================
# 1) Upload zip & extract automatically
# =========================================
print("📂 Please upload your wound dataset ZIP (class folders inside):")
uploaded = files.upload()
if len(uploaded) == 0:
    raise RuntimeError("No file uploaded.")

zip_filename = list(uploaded.keys())[0]
zip_path = f"/content/{zip_filename}"
extract_root = "/content/wound_dataset_extracted"

# Clean any previous extraction
if os.path.exists(extract_root):
    shutil.rmtree(extract_root)
os.makedirs(extract_root, exist_ok=True)

# Try Python ZipFile first, fallback to 7z if needed
extracted_ok = False
try:
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_root)
    print("✅ Dataset extracted successfully with Python zipfile.")
    extracted_ok = True
except Exception as e:
    print("⚠️ zipfile failed, trying 7z instead:", e)

if not extracted_ok:
    print("📦 Installing 7zip and extracting...")
    !apt-get -y install -qq p7zip-full
    !7z x "{zip_path}" -o"{extract_root}" -y
    print("✅ Dataset extracted successfully with 7z.")

print("\nTop-level items under extract_root:")
print(os.listdir(extract_root))

# =========================================
# 2) Auto-detect base_dir (folder that has class folders)
# =========================================
def find_base_dir(root):
    """
    Heuristic:
    - If root directly contains many subfolders and few files -> treat root as base_dir.
    - Else if there's exactly ONE subfolder, treat that subfolder as base_dir.
    """
    entries = os.listdir(root)
    subdirs = [d for d in entries if os.path.isdir(os.path.join(root, d))]
    files_ = [f for f in entries if os.path.isfile(os.path.join(root, f))]

    # If root already looks like a class-folder root
    if len(subdirs) >= 2:
        return root

    # If exactly one subfolder, go one level down
    if len(subdirs) == 1:
        candidate = os.path.join(root, subdirs[0])
        print(f"🔎 Using single subfolder as base_dir: {candidate}")
        return candidate

    # Fallback
    return root

base_dir = find_base_dir(extract_root)
print("\n✅ Chosen base_dir:", base_dir)

categories = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
print("Detected wound categories:", categories)

if len(categories) < 2:
    raise RuntimeError("Not enough class folders detected. Your ZIP should contain folders per wound class.")

# =========================================
# 3) Train/test split into dedicated folders
# =========================================
train_dir = "/content/wound_dataset_split/train"
test_dir  = "/content/wound_dataset_split/test"

for d in [train_dir, test_dir]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

print("\n📁 Splitting into train/test...")

for category in categories:
    category_path = os.path.join(base_dir, category)
    if not os.path.isdir(category_path):
        continue

    images = [f for f in os.listdir(category_path)
              if os.path.isfile(os.path.join(category_path, f))
              and f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]

    if len(images) == 0:
        print(f"⚠️ Skipping empty category: {category}")
        continue

    train_imgs, test_imgs = train_test_split(images, test_size=0.2, random_state=SEED)

    os.makedirs(os.path.join(train_dir, category), exist_ok=True)
    os.makedirs(os.path.join(test_dir, category), exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(category_path, img),
                    os.path.join(train_dir, category, img))
    for img in test_imgs:
        shutil.copy(os.path.join(category_path, img),
                    os.path.join(test_dir, category, img))

print("✅ Data split into train/test successfully!")
print("Train classes:", os.listdir(train_dir))
print("Test classes:", os.listdir(test_dir))

# =========================================
# 4) Image generators (no augmentation)
# =========================================
IMG_SIZE = (150, 150)
BATCH    = 32

data_gen = ImageDataGenerator(rescale=1./255)

train_generator = data_gen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_gen = data_gen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode="categorical",
    shuffle=False
)

NUM_CLASSES = train_generator.num_classes
CLASS_NAMES = list(train_generator.class_indices.keys())
print("\nClasses found:", CLASS_NAMES)
print("Number of wound classes:", NUM_CLASSES)

# =========================================
# 5) Class-level detailed symptom lists
#    (class name is normalized, e.g., 'Burns' -> 'burns')
# =========================================
detailed_symptoms = {
    "abrasion": [
        "superficial skin loss with raw red surface",
        "mild oozing of blood from scraped area",
        "burning pain on touching or movement",
        "surrounding redness but limited depth",
        "often caused by fall or friction on hard surface"
    ],
    "abrasions": [
        "superficial skin loss with raw red surface",
        "mild oozing of blood from scraped area",
        "burning pain on touching or movement",
        "surrounding redness but limited depth",
        "often caused by fall or friction on hard surface"
    ],
    "bruise": [
        "purple or blue skin discoloration",
        "tenderness when pressed",
        "swelling around injured area",
        "no open skin break but visible color change",
        "gradual color change from red to purple to yellow"
    ],
    "bruises": [
        "purple or blue skin discoloration",
        "tenderness when pressed",
        "swelling around injured area",
        "no open skin break but visible color change",
        "gradual color change from red to purple to yellow"
    ],
    "burn": [
        "redness and blistering of skin",
        "severe burning pain",
        "fluid filled blisters that may rupture",
        "skin may appear white or charred in deep burns",
        "swelling around the burned area"
    ],
    "burns": [
        "redness and blistering of skin",
        "severe burning pain",
        "fluid filled blisters that may rupture",
        "skin may appear white or charred in deep burns",
        "swelling around the burned area"
    ],
    "laceration": [
        "deep cut with irregular wound edges",
        "visible separation of skin layers",
        "active bleeding from wound",
        "pain on movement or touch",
        "may require suturing to close edges"
    ],
    "lacerations": [
        "deep cut with irregular wound edges",
        "visible separation of skin layers",
        "active bleeding from wound",
        "pain on movement or touch",
        "may require suturing to close edges"
    ],
    "incision": [
        "clean straight cut often from sharp object",
        "edges of wound are smooth and well defined",
        "bleeding present but usually controlled",
        "pain localized to wound line",
        "often related to surgical procedure or knife injury"
    ],
    "incisions": [
        "clean straight cut often from sharp object",
        "edges of wound are smooth and well defined",
        "bleeding present but usually controlled",
        "pain localized to wound line",
        "often related to surgical procedure or knife injury"
    ],
    "ulcer": [
        "open sore with loss of skin and underlying tissue",
        "slow to heal and sometimes chronic",
        "may show yellow slough or necrotic tissue",
        "surrounding skin may be dark or discolored",
        "may have foul smell or discharge"
    ],
    "ulcers": [
        "open sore with loss of skin and underlying tissue",
        "slow to heal and sometimes chronic",
        "may show yellow slough or necrotic tissue",
        "surrounding skin may be dark or discolored",
        "may have foul smell or discharge"
    ],
    "pressureulcer": [
        "wound over bony prominence due to prolonged pressure",
        "skin breakdown with red or dark patches",
        "may progress to deep crater like wound",
        "surrounding area may feel warmer or harder",
        "often seen in bedridden patients"
    ],
    "diabeticwound": [
        "chronic wound on foot or lower leg in diabetic patient",
        "reduced pain despite open wound",
        "slow or non healing ulcer",
        "surrounding skin may be dry cracked or dark",
        "possible infection with discharge or smell"
    ],
    "surgicalwound": [
        "linear incision with sutures or staples",
        "edges of wound closely opposed",
        "mild redness and swelling around stitch line",
        "usually less pain at rest more with movement",
        "risk of dehiscence or infection if not cared"
    ],
    "penetrating": [
        "small entry wound with deeper tissue damage",
        "pain out of proportion to visible size",
        "possible bleeding from deeper structures",
        "may be caused by nail metal or sharp object",
        "risk of internal organ or tissue injury"
    ]
}

UNIVERSAL_SYMPTOMS = [
    "redness",
    "pain",
    "swelling",
    "blistering",
    "bleeding",
    "bruising",
    "open wound",
    "skin loss",
    "discoloration",
    "fluid discharge",
    "foul smell",
    "burning",
    "tenderness",
    "necrotic tissue",
    "slow healing"
]

keyword_map = {
    "red": "redness", "redness": "redness", "inflam": "redness",
    "warm": "redness", "hot": "redness",

    "pain": "pain", "painful": "pain", "tender": "tenderness",
    "sore": "pain", "hurts": "pain",
    "burn": "burning", "burning": "burning",

    "swel": "swelling", "swelling": "swelling", "puffy": "swelling",

    "blister": "blistering", "bubble": "blistering",
    "vesicle": "blistering", "fluid filled": "blistering",

    "bleed": "bleeding", "blood": "bleeding", "oozing": "bleeding",

    "bruise": "bruising", "bruising": "bruising", "purple": "bruising",
    "blue": "bruising", "black": "discoloration", "dark": "discoloration",
    "discolor": "discoloration",

    "open": "open wound", "deep": "open wound", "cut": "open wound",
    "gaping": "open wound", "skin loss": "skin loss",

    "pus": "fluid discharge", "discharge": "fluid discharge",
    "smell": "foul smell", "odor": "foul smell",

    "slough": "necrotic tissue", "necrotic": "necrotic tissue",

    "chronic": "slow healing", "slow": "slow healing", "non healing": "slow healing"
}

def map_to_universal(phrase):
    phrase_l = phrase.lower()
    for k, v in keyword_map.items():
        if k in phrase_l:
            return v
    return random.choice(UNIVERSAL_SYMPTOMS)

# =========================================
# 6) Build sym_df: map each image file -> one universal token
# =========================================
rows = []
for root in [train_dir, test_dir]:
    for folder in sorted(os.listdir(root)):
        folder_path = os.path.join(root, folder)
        if not os.path.isdir(folder_path):
            continue
        key = re.sub(r'[^a-z0-9]+','', folder.lower())
        detailed_list = detailed_symptoms.get(key, [])
        imgs = sorted([
            f for f in os.listdir(folder_path)
            if f.lower().endswith(('jpg','jpeg','png','webp'))
        ])
        for i, fname in enumerate(imgs):
            if detailed_list:
                phrase = detailed_list[i % len(detailed_list)]
                uni = map_to_universal(phrase)
            else:
                uni = random.choice(UNIVERSAL_SYMPTOMS)
            rel = os.path.join(folder, fname)
            rows.append([rel, uni])

sym_df = pd.DataFrame(rows, columns=['relpath','symptoms'])
sym_df.to_csv('wound_symptoms.csv', index=False)
print("\n📝 wound_symptoms.csv saved, rows =", len(sym_df))

# =========================================
# 7) Build wound CNN
# =========================================
model = Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(512, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Conv2D(512, (3,3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(512, activation="relu", name="image_features"),
    Dropout(0.6),
    Dense(128, activation="relu"),
    Dropout(0.6),
    Dense(NUM_CLASSES, activation="softmax")
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

# =========================================
# 8) Train wound CNN
# =========================================
EPOCHS_CNN = 25
history = model.fit(
    train_generator,
    epochs=EPOCHS_CNN,
    validation_data=val_gen,
    verbose=1
)

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history.get('accuracy',[]), label='train')
plt.plot(history.history.get('val_accuracy',[]), label='val')
plt.legend(); plt.title('Accuracy')

plt.subplot(1,2,2)
plt.plot(history.history.get('loss',[]), label='train')
plt.plot(history.history.get('val_loss',[]), label='val')
plt.legend(); plt.title('Loss')
plt.show()

cnn_test_loss, cnn_test_acc = model.evaluate(val_gen, verbose=0)
print(f'\n📊 Wound CNN baseline accuracy: {cnn_test_acc*100:.2f}%')

# =========================================
# 9) Build feature extractor
# =========================================
_ = model.predict(np.zeros((1, IMG_SIZE[0], IMG_SIZE[1], 3)))
feature_extractor = Model(
    inputs=model.layers[0].input,
    outputs=model.get_layer('image_features').output
)
print("✅ Feature extractor ready")

# =========================================
# 10) Ordered generators for embeddings
# =========================================
ordered_train = ImageDataGenerator(rescale=1./255).flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode='categorical', shuffle=False)

ordered_val = ImageDataGenerator(rescale=1./255).flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode='categorical', shuffle=False)

train_steps = math.ceil(ordered_train.samples / BATCH)
val_steps   = math.ceil(ordered_val.samples / BATCH)

# =========================================
# 11) Extract image embeddings
# =========================================
train_img_emb = feature_extractor.predict(ordered_train, steps=train_steps, verbose=1)
val_img_emb   = feature_extractor.predict(ordered_val,   steps=val_steps,   verbose=1)
print("Image embeddings shapes:", train_img_emb.shape, val_img_emb.shape)

# =========================================
# 12) Text inputs for SBERT
# =========================================
sym_map = dict(zip(sym_df.relpath, sym_df.symptoms))
train_texts = [sym_map.get(fp, random.choice(UNIVERSAL_SYMPTOMS)) for fp in ordered_train.filenames]
val_texts   = [sym_map.get(fp, random.choice(UNIVERSAL_SYMPTOMS)) for fp in ordered_val.filenames]

def add_generic_dropout(texts, drop_frac=0.10):
    out = []
    for t in texts:
        if random.random() < drop_frac:
            out.append("unspecified wound complaint")
        else:
            out.append(t)
    return out

train_texts_aug = add_generic_dropout(train_texts, drop_frac=0.10)

# =========================================
# 13) Text embeddings with SBERT
# =========================================
text_model = SentenceTransformer('all-MiniLM-L6-v2')
train_text_emb = text_model.encode(train_texts_aug, batch_size=32,
                                   show_progress_bar=True, convert_to_numpy=True)
val_text_emb   = text_model.encode(val_texts, batch_size=32,
                                   show_progress_bar=True, convert_to_numpy=True)
print("Text embeddings shapes:", train_text_emb.shape, val_text_emb.shape)

# =========================================
# 14) Labels
# =========================================
y_train = to_categorical(ordered_train.labels, num_classes=NUM_CLASSES)
y_val   = to_categorical(ordered_val.labels,   num_classes=NUM_CLASSES)

# =========================================
# 15) Fusion model
# =========================================
img_dim = train_img_emb.shape[1]
txt_dim = train_text_emb.shape[1]

img_in = Input(shape=(img_dim,), name='img_in')
txt_in = Input(shape=(txt_dim,), name='txt_in')

txt_proj = Dense(img_dim, activation='relu', name='txt_proj')(txt_in)
txt_proj = BatchNormalization()(txt_proj)
txt_proj = Dropout(0.2)(txt_proj)

fusion = Concatenate(name='fusion')([img_in, txt_proj])
x = Dense(512, activation='relu')(fusion); x = BatchNormalization()(x); x = Dropout(0.3)(x)
x = Dense(256, activation='relu')(x);     x = BatchNormalization()(x); x = Dropout(0.2)(x)
x = Dense(128, activation='relu')(x);     x = Dropout(0.15)(x)
out = Dense(NUM_CLASSES, activation='softmax')(x)

fusion_model = Model([img_in, txt_in], out, name='wound_fusion_model')
fusion_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
                     loss='categorical_crossentropy', metrics=['accuracy'])
fusion_model.summary()

# =========================================
# 16) Train fusion
# =========================================
fusion_callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7,
                                     restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                         patience=4, min_lr=1e-7),
    tf.keras.callbacks.ModelCheckpoint('/content/best_wound_fusion.h5',
                                       monitor='val_accuracy', save_best_only=True, mode='max')
]

history_f = fusion_model.fit(
    [train_img_emb, train_text_emb], y_train,
    validation_data=([val_img_emb, val_text_emb], y_val),
    epochs=30, batch_size=16, callbacks=fusion_callbacks, verbose=1
)

# =========================================
# 17) Evaluate fusion
# =========================================
test_loss, test_acc = fusion_model.evaluate([val_img_emb, val_text_emb], y_val, verbose=0)
print(f'\n🧠 Wound Fusion test accuracy: {test_acc*100:.2f}% (CNN baseline {cnn_test_acc*100:.2f}%)')

pred_probs = fusion_model.predict([val_img_emb, val_text_emb])
pred_labels = np.argmax(pred_probs, axis=1)
true_labels = np.argmax(y_val, axis=1)
class_names = list(ordered_train.class_indices.keys())

cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Wound Fusion Confusion Matrix')
plt.show()

print("\nClassification Report:\n")
print(classification_report(true_labels, pred_labels,
                            target_names=class_names, digits=4))

# =========================================
# 18) Save artifacts
# =========================================
fusion_model.save('/content/wound_fusion_model.h5')
model.save('/content/wound_cnn_image_model.h5')
model.save('/content/wound_cnn_image_model.keras')
feature_extractor.save('/content/wound_feature_extractor.h5')
joblib.dump(class_names, '/content/wound_class_names.pkl')
joblib.dump(sym_map, '/content/wound_sym_map.pkl')

print("\n✅ Saved models & artifacts to /content:")
print(" - /content/wound_fusion_model.h5")
print(" - /content/wound_cnn_image_model.h5")
print(" - /content/wound_cnn_image_model.keras")
print(" - /content/wound_feature_extractor.h5")
print(" - /content/wound_class_names.pkl")
print(" - /content/wound_sym_map.pkl")

# =========================================
# 19) Multimodal inference helper
# =========================================
from tensorflow.keras.preprocessing import image as kimg

def predict_wound_multimodal(image_path, symptoms_text):
    """
    image_path: local path to wound image file
    symptoms_text: free-text symptom description provided by user
    """
    img = kimg.load_img(image_path, target_size=IMG_SIZE)
    arr = kimg.img_to_array(img) / 255.0
    arr = np.expand_dims(arr, 0)
    img_emb = feature_extractor.predict(arr, verbose=0)

    txt = symptoms_text.lower()
    mapped = None
    for k, v in keyword_map.items():
        if k in txt:
            mapped = v
            break
    if mapped is None:
        mapped = "unspecified wound complaint"
    txt_emb = text_model.encode([mapped], convert_to_numpy=True)

    probs = fusion_model.predict([img_emb, txt_emb], verbose=0)[0]
    idx = int(np.argmax(probs))
    return class_names[idx], float(probs[idx]), probs

print("\n🎉 Inference ready. Example call:")
print("predict_wound_multimodal('/content/wound_dataset_split/test/<CLASS>/<image>.jpg', 'redness swelling and pain')")


In [ ]:
from google.colab import files

files.download("/content/wound_fusion_model.h5")
files.download("/content/wound_feature_extractor.h5")
files.download("/content/wound_class_names.pkl")
